# 01 Prepare Inputs

Prepare static input assets used by the AMR workflow.

Inputs/outputs live in `assets/`. This notebook validates or creates:

- `assets/counts_extended2.csv`: ranked pathogen priority list from Zenodo.
- `assets/antibiotic_synonym_lookup.csv`: antibiotic synonym lookup.
- `assets/antibiotic_coarsening_lookup.csv`: canonical antibiotic category lookup.

## Inputs

- Curated pathogen-priority data and the pathogen and antibiotic synonym/coarsening sources used by this workflow.

## Outputs

- Validated or generated lookup tables in `assets/`, including the pathogen priority list and antibiotic synonym and coarsening tables.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## Configuration


In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ASSETS_DIR = PROJECT_ROOT / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

PATHOGEN_PRIORITY_URL = "https://zenodo.org/records/18376711/files/counts_extended2.csv?download=1"
PATHOGEN_PRIORITY_OUTPUT = ASSETS_DIR / "counts_extended2.csv"
PATHOGEN_PRIORITY_REQUIRED_COLS = {"CanonicalSpecies", "pathogenic_index"}

SOURCE_XLSX = ASSETS_DIR / "antibiotic_list3.xlsx"
SOURCE_SHEET = "Sheet1"

ANTIBIOTIC_SYNONYM_OUTPUT = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
ANTIBIOTIC_COARSENING_OUTPUT = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"

CANONICAL_COL = "Antibiotic"
SYNONYM_COLS = ["alternative spelling", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]
COARSENING_COLS = [
    "Antibiotic",
    "Class",
    "Simplified categories",
    "High level categories",
    "Origin",
]





## Prepare Pathogen Priority List

Download the ranked pathogen list if it is not already present in `assets/`, then validate the columns used by collection.


In [3]:
if not PATHOGEN_PRIORITY_OUTPUT.exists():
    print(f"Downloading pathogen priority list to {PATHOGEN_PRIORITY_OUTPUT}")
    pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_URL)
    pathogen_priority_df.to_csv(PATHOGEN_PRIORITY_OUTPUT, index=False)
else:
    pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_OUTPUT)

missing_priority_cols = PATHOGEN_PRIORITY_REQUIRED_COLS - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_OUTPUT} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"], errors="coerce"
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[pathogen_priority_df["CanonicalSpecies"] != ""].copy()
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"], ascending=[False, True]
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")
pathogen_priority_df.to_csv(PATHOGEN_PRIORITY_OUTPUT, index=False)

print(f"Saved {len(pathogen_priority_df):,} pathogen priority rows to {PATHOGEN_PRIORITY_OUTPUT}")
display(pathogen_priority_df.head(10))


Saved 1,279 pathogen priority rows to /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/counts_extended2.csv


,CanonicalSpecies,n_causation,n_articles,n_sampled,pathogenic_index,PMID
1071,Salmonella enterica,554,24694,200,1.000000,"2359909,8229147,27938338,6675986,30394218,9951..."
786,Mycobacterium tuberculosis,299,20559,200,0.928144,"9929731,27847025,15200832,35242163,25569290,20..."
282,Clostridioides difficile,298,17012,200,0.910833,"39355263,31548637,33110112,35663882,34856509,3..."
494,Haemophilus influenzae,200,14537,200,0.860892,"7224344,6791566,6097126,988430,13051770,132222..."
546,Klebsiella pneumoniae,176,15847,200,0.857160,"721945,25799383,22669526,20514860,17725256,272..."
1187,Streptococcus pneumoniae,122,13743,200,0.811449,"23198578,33541757,23186827,37120991,27892542,2..."
518,Helicobacter pylori,96,15542,200,0.800970,"2814348,3072666,2755371,2642447,10150401,26375..."
193,Brucella melitensis,258,5214,200,0.791666,"31966957,27765819,18178848,37853407,32234817,3..."
158,Borrelia burgdorferi,151,7692,200,0.778476,"34923426,29272385,31308087,29026617,26356494,1..."
998,Pseudomonas aeruginosa,63,17254,200,0.772522,"39200075,24820475,34115807,24603807,37007650,2..."


In [16]:
pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_OUTPUT)
pathogen_priority_df[pathogen_priority_df['n_causation']>=3]

,CanonicalSpecies,n_causation,n_articles,n_sampled,pathogenic_index,PMID
0,Salmonella enterica,554,24694,200,1.000000,"2359909,8229147,27938338,6675986,30394218,9951..."
1,Mycobacterium tuberculosis,299,20559,200,0.928144,"9929731,27847025,15200832,35242163,25569290,20..."
2,Clostridioides difficile,298,17012,200,0.910833,"39355263,31548637,33110112,35663882,34856509,3..."
3,Haemophilus influenzae,200,14537,200,0.860892,"7224344,6791566,6097126,988430,13051770,132222..."
4,Klebsiella pneumoniae,176,15847,200,0.857160,"721945,25799383,22669526,20514860,17725256,272..."
...,...,...,...,...,...,...
776,Streptococcus lutetiensis,3,19,19,0.098679,"34712909,39568311,31157186"
777,Streptomyces albus,3,36,36,0.098679,"15583353,22477566,5418302"
778,Veillonella atypica,3,41,41,0.098679,"34861365,31576208,36746223"
779,Vibrio cincinnatiensis,3,8,8,0.098679,"3510054,2422196,35845826"


In [21]:
OLD_FILE = ASSETS_DIR / "fabrizio_list.xlsx"
old_pathogen_df = pd.read_excel(OLD_FILE)
old_pathogen_df[old_pathogen_df['n_causation']>=3]

,CanonicalSpecies,n_causation,n_articles,n_sampled,pathogenic_index
0,Salmonella enterica,397,24762,1268,0.98
1,Mycobacterium tuberculosis,222,20559,805,0.94
2,Escherichia coli,33,18255,200,0.87
3,Pseudomonas aeruginosa,38,17254,200,0.88
4,Staphylococcus aureus,42,17240,200,0.89
...,...,...,...,...,...
987,Suttonella indologenes,3,5,5,0.12
1001,Gordonia otitidis,3,4,4,0.12
1009,Mycobacterium arosiense,3,4,4,0.12
1023,Pseudomonas citronellolis,3,4,4,0.12


In [ ]:
differences_list= set(pathogen_priority_df["CanonicalSpecies"].dropna()) ^ set(old_pathogen_df["CanonicalSpecies"].dropna())
#len(differences_list)
differences_list

{'Achromobacter insolitus',
 'Achromobacter spanius',
 'Acidaminococcus intestini',
 'Acinetobacter gyllenbergii',
 'Acinetobacter lactucae',
 'Actinomadura bangladeshensis',
 'Aerococcus urinaehominis',
 'Afipia felis',
 'Alloiococcus otitis',
 'Bacillus paranthracis',
 'Bacillus toyonensis',
 'Bacteroides heparinolyticus',
 'Blastomonas natatoria',
 'Brevibacillus agri',
 'Brucella tritici',
 'Burkholderia latens',
 'Capnocytophaga granulosa',
 'Catonella morbi',
 'Caulobacter vibrioides',
 'Chryseomonas luteola',
 'Citrobacter diversus',
 'Citrobacter youngae',
 'Corynebacterium belfantii',
 'Desulfovibrio porci',
 'Ehrlichia canis',
 'Ehrlichia japonica',
 'Enterobacter quasiroggenkampii',
 'Enterocloster aldenensis',
 'Enterococcus thailandicus',
 'Escherichia hermannii',
 'Eubacterium limosum',
 'Flavobacterium devorans',
 'Fusobacterium periodonticum',
 'Haematomicrobium sanguinis',
 'Halomonas johnsoniae',
 'Helicobacter mustelae',
 'Heyndrickxia coagulans',
 'Hoylesella loesch

: 

## Load and Validate Source Workbook


In [4]:
source_df = pd.read_excel(SOURCE_XLSX, sheet_name=SOURCE_SHEET)
source_df.columns = [str(col).strip() for col in source_df.columns]

required_cols = {CANONICAL_COL, *SYNONYM_COLS, *COARSENING_COLS}
missing_cols = required_cols - set(source_df.columns)
if missing_cols:
    raise KeyError(f"{SOURCE_XLSX} is missing required columns: {sorted(missing_cols)}")

source_df = source_df.dropna(subset=[CANONICAL_COL]).copy()
source_df[CANONICAL_COL] = source_df[CANONICAL_COL].astype(str).str.strip()
source_df = source_df.loc[source_df[CANONICAL_COL] != ""].copy()

print(f"Loaded {len(source_df):,} antibiotic rows from {SOURCE_XLSX}")
display(source_df.head(10))


Loaded 245 antibiotic rows from /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_list3.xlsx


,Antibiotic,alternative spelling,Unnamed: 2,Unnamed: 3,Unnamed: 4,Class,Simplified categories,High level categories,Origin
0,Amikacin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Arbekacin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
2,Bekanamycin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
3,Dibekacin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Gentamicin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
5,Isepamicin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
6,Kanamycin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
7,Micronomicin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
8,Neomycin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
9,Netilmicin,NaN,NaN,NaN,NaN,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified


## Build Antibiotic Synonym Lookup

The output mirrors the bacteria synonym lookup shape: one synonym row per canonical antibiotic. The canonical antibiotic name is included as its own synonym so downstream query expansion can use a single lookup table.


In [5]:
def clean_text(value):
    if pd.isna(value):
        return None
    text = " ".join(str(value).strip().split())
    return text or None


synonym_rows = []
for _, row in source_df.iterrows():
    canonical = clean_text(row[CANONICAL_COL])
    if canonical is None:
        continue
    variants = [canonical]
    variants.extend(clean_text(row[col]) for col in SYNONYM_COLS)
    for variant in variants:
        if variant is not None:
            synonym_rows.append(
                {
                    "antibiotic synonym": variant,
                    "antibiotic name": canonical,
                }
            )

antibiotic_synonym_df = (
    pd.DataFrame(synonym_rows)
    .drop_duplicates()
    .sort_values(["antibiotic name", "antibiotic synonym"])
    .reset_index(drop=True)
)

antibiotic_synonym_df.to_csv(ANTIBIOTIC_SYNONYM_OUTPUT, index=False)
print(f"Saved {len(antibiotic_synonym_df):,} synonym rows to {ANTIBIOTIC_SYNONYM_OUTPUT}")
display(antibiotic_synonym_df.head(20))


Saved 422 synonym rows to /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_synonym_lookup.csv


,antibiotic synonym,antibiotic name
0,Amikacin,Amikacin
1,Amoxicillin,Amoxicillin
2,Amoxicillin-clavulanic-acid,Amoxicillin/clavulanic-acid
3,Amoxicillin/clavulanic-acid,Amoxicillin/clavulanic-acid
4,Amoxicyllin-clavulanic-acid,Amoxicillin/clavulanic-acid
5,Amoxicyllin/clavulanic-acid,Amoxicillin/clavulanic-acid
6,Ampicillin,Ampicillin
7,Ampicillin-sulbactam,Ampicillin/sulbactam
8,Ampicillin/sulbactam,Ampicillin/sulbactam
9,Arbekacin,Arbekacin


## Build Antibiotic Coarsening Lookup

This table stores the canonical antibiotic-to-category mapping used for grouped analyses and coarser visualizations.


In [6]:
antibiotic_coarsening_df = source_df[COARSENING_COLS].copy()
for col in COARSENING_COLS:
    antibiotic_coarsening_df[col] = antibiotic_coarsening_df[col].map(clean_text)

antibiotic_coarsening_df = (
    antibiotic_coarsening_df
    .dropna(subset=["Antibiotic"])
    .drop_duplicates(subset=["Antibiotic"], keep="first")
    .sort_values("Antibiotic")
    .reset_index(drop=True)
)

antibiotic_coarsening_df.to_csv(ANTIBIOTIC_COARSENING_OUTPUT, index=False)
print(f"Saved {len(antibiotic_coarsening_df):,} coarsening rows to {ANTIBIOTIC_COARSENING_OUTPUT}")
display(antibiotic_coarsening_df.head(20))


Saved 245 coarsening rows to /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_coarsening_lookup.csv


,Antibiotic,Class,Simplified categories,High level categories,Origin
0,Amikacin,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Amoxicillin,Penicillins,Penicillins,Beta lactam and related,Naturally occurring/modified
2,Amoxicillin/clavulanic-acid,Beta-lactam/beta-lactamase-inhibitor,Beta-lactamase-inhibitors/combinations,Beta lactam and related,Naturally occurring/modified
3,Ampicillin,Penicillins,Penicillins,Beta lactam and related,Naturally occurring/modified
4,Ampicillin/sulbactam,Beta-lactam/beta-lactamase-inhibitor,Beta-lactamase-inhibitors/combinations,Beta lactam and related,Naturally occurring/modified
5,Arbekacin,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
6,Aspoxicillin,Penicillins,Penicillins,Beta lactam and related,Naturally occurring/modified
7,Azidocillin,Penicillins,Penicillins,Beta lactam and related,Naturally occurring/modified
8,Azithromycin,Macrolides,Macrolides,MLS,Naturally occurring/modified
9,Azlocillin,Penicillins,Penicillins,Beta lactam and related,Naturally occurring/modified


## Quick Checks


In [7]:
print("Canonical antibiotics:", antibiotic_coarsening_df["Antibiotic"].nunique())
print("Synonym variants:", antibiotic_synonym_df["antibiotic synonym"].nunique())
print("Classes:", antibiotic_coarsening_df["Class"].nunique(dropna=True))
print("Simplified categories:", antibiotic_coarsening_df["Simplified categories"].nunique(dropna=True))
print("High level categories:", antibiotic_coarsening_df["High level categories"].nunique(dropna=True))

duplicate_synonyms = antibiotic_synonym_df.groupby("antibiotic synonym")["antibiotic name"].nunique()
duplicate_synonyms = duplicate_synonyms[duplicate_synonyms > 1]
if duplicate_synonyms.empty:
    print("No synonym variants map to multiple canonical antibiotics.")
else:
    print("Synonym variants mapping to multiple canonical antibiotics:")
    display(duplicate_synonyms.reset_index(name="canonical_count"))


Canonical antibiotics: 245
Synonym variants: 422
Classes: 38
Simplified categories: 15
High level categories: 11
No synonym variants map to multiple canonical antibiotics.
